# Preprocesamiento de Datos

Limpieza, balanceo y selección de columnas del dataset .

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

spark = SparkSession.builder.appName("Preprocessing_Fraud").getOrCreate()

df_raw = spark.read.csv("../data/raw/bank_fraud.csv", header=True, inferSchema=True)

print(f"Total de registros crudos: {df_raw.count()}")
df_raw.printSchema()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 18:41:29 WARN Utils: Your hostname, zamallita-HP-Notebook, resolves to a loopback address: 127.0.1.1; using 192.168.100.40 instead (on interface wlp2s0)
26/06/14 18:41:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 18:41:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 18:41:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Total de registros crudos: 1000000
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night_transaction: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- account_age_years: double (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- num_prev_transactions: integer (nullable = true)
 |-- transaction_freq_monthly: integer (nullable = true)
 |-- distance_from_home_km: double (nullable = true)
 |-- time_since_last_

## 1. Eliminar filas con valores nulos

In [2]:
# Eliminamos filas donde haya nulos en las columnas que usaremos
df_clean = df_raw.dropna()

print(f"Registros después de quitar nulos: {df_clean.count()}")


26/06/14 18:42:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Registros después de quitar nulos: 1000000


## 2. Selección de columnas

**Columnas numéricas (15):** todas las disponibles para no perder información.

**Columnas categóricas (text):** , , 

**Variable objetivo:** 

In [3]:
NUMERIC_COLS = [
    "hour_of_day",
    "is_weekend",
    "is_night_transaction",
    "customer_age",
    "credit_score",
    "account_age_years",
    "account_balance",
    "transaction_amount",
    "num_prev_transactions",
    "transaction_freq_monthly",
    "distance_from_home_km",
    "time_since_last_txn_hrs",
    "is_international",
    "failed_attempts",
    "pin_changed_recently"
]

CATEGORICAL_COLS = [
    "merchant_category",
    "payment_method",
    "device_type"
]

TARGET_COL = "is_fraud"

selected_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
df_selected = df_clean.select(selected_cols)

print(f"Columnas seleccionadas: {df_selected.columns}")


Columnas seleccionadas: ['hour_of_day', 'is_weekend', 'is_night_transaction', 'customer_age', 'credit_score', 'account_age_years', 'account_balance', 'transaction_amount', 'num_prev_transactions', 'transaction_freq_monthly', 'distance_from_home_km', 'time_since_last_txn_hrs', 'is_international', 'failed_attempts', 'pin_changed_recently', 'merchant_category', 'payment_method', 'device_type', 'is_fraud']


## 3. Balanceo de clases (Undersampling)

El dataset está muy desbalanceado: ~55k fraudes vs ~944k no-fraudes.
Aplicamos undersampling con ratio **1:3** (fraude:no fraude) para reducir el dataset a ~220k filas,
conservando representatividad sin sobrecargar el entrenamiento.

Cambia  para ajustar el balance:  
-  → 50/50 balanceado (~110k filas)  
-  → 1:3 recomendado (~220k filas)  
-  → 1:5 más realista (~330k filas)

In [4]:
RATIO_NO_FRAUD = 3  # Por cada fraude, cuántos no-fraude conservar

# Separar por clase
df_fraud = df_selected.filter(col("is_fraud") == 1)
df_no_fraud = df_selected.filter(col("is_fraud") == 0)

n_fraud = df_fraud.count()
n_no_fraud_target = n_fraud * RATIO_NO_FRAUD
total_no_fraud = df_no_fraud.count()

print(f"Fraudes encontrados: {n_fraud}")
print(f"No-fraudes totales: {total_no_fraud}")
print(f"No-fraudes a conservar (ratio 1:{RATIO_NO_FRAUD}): {n_no_fraud_target}")

# Calcular fracción para el sample
fraction = min(n_no_fraud_target / total_no_fraud, 1.0)
df_no_fraud_sampled = df_no_fraud.sample(withReplacement=False, fraction=fraction, seed=42)

# Unir ambas clases
df_balanced = df_fraud.union(df_no_fraud_sampled)
df_balanced = df_balanced.orderBy(col("is_fraud").desc())  # Shuffle básico

total = df_balanced.count()
print(f"Dataset final: {total} filas")
print(f"  - Fraude: {df_fraud.count()} ({df_fraud.count()/total*100:.1f}%)")
print(f"  - No fraude: {df_no_fraud_sampled.count()} ({df_no_fraud_sampled.count()/total*100:.1f}%)")


Fraudes encontrados: 55255
No-fraudes totales: 944745
No-fraudes a conservar (ratio 1:3): 165765


Dataset final: 221068 filas


  - Fraude: 55255 (25.0%)


  - No fraude: 165813 (75.0%)


## 4. Guardar datos procesados

In [5]:
# Guardar como CSV en data/processed/
output_path = "../data/processed/bank_fraud_clean.csv"

df_balanced.coalesce(1).write.csv(output_path, header=True, mode="overwrite")

print(f"Dataset guardado en: {output_path}")
print(f"Total de filas guardadas: {df_balanced.count()}")


Dataset guardado en: ../data/processed/bank_fraud_clean.csv


Total de filas guardadas: 221068
